# Dataset statistics

Count and group-by queries with matplotlib charts. Use this
notebook to get a feel for how the corpus is distributed across
agent types, ego actions, and environments before you go
looking for specific scenarios.

> See `01_quickstart.ipynb` for an introduction to the dataset
> and `02_query_dsl_tour.ipynb` for the query language itself.


## Setup


In [ ]:
import os
from collections import Counter
from pathlib import Path

from cascade_av.dataset import CascadeDataset

# --- shared notebook styling -----------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
    "figure.dpi": 110,
    "savefig.dpi": 110,
})

pd.options.display.max_colwidth = 110
pd.options.display.width = 140
pd.options.display.float_format = "{:.2f}".format


In [ ]:
ds = CascadeDataset(Path(os.environ["CASCADE_AV_DATASET_ROOT"]))
print(f"corpus loaded — {len(ds)} clips")

def entity_count(query: str) -> int:
    """Number of matching *entities* across the corpus (not clips)."""
    return len(ds.find(query).matches)

def entities_by_attr(query: str, attr: str) -> dict:
    """Bucket matching entities by a top-level attribute (e.g. ``type``).

    Like ``ds.group_by`` but counts entities rather than clips.
    """
    out = Counter()
    for m in ds.find(query).matches:
        val = getattr(m.entity, attr, None)
        if isinstance(val, list):
            for v in val:
                out[v] += 1
        else:
            out[val] += 1
    return dict(out)

def actions_with_flag(query: str, flag: str) -> int:
    """Number of matching *action* entities where ``flag`` is True.

    Use with action-scoped queries (e.g. ``agent(type=ped,
    action(jaywalk=true))``). Flag examples: ``jaywalk``,
    ``erratic``, ``aggressive``, ``illegal``, ``aborted``,
    ``unprotected``.
    """
    return sum(
        1 for m in ds.find(query).matches
        if getattr(m.entity, flag, False) is True
    )


## Clips vs entities

The query API exposes two complementary perspectives:

- **`ds.count(query)`** — the number of *clips* with ≥1 match.
  Tells you about *coverage*: "how many clips even contain
  this thing?"
- **`len(ds.find(query).matches)`** — the number of *matching
  entities*. Tells you about *prevalence*: "how often does
  this thing occur?" A single clip can contribute many
  entities (six pedestrians, four vehicles, …).

Both are useful — coverage tells you how varied the corpus
is; prevalence tells you the actual instance counts.


## Headline counts


In [ ]:
primitives = [
    ("pedestrian",              "agent.type = ped"),
    ("vehicle",                 "agent.type = vehicle"),
    ("VRU (ped/cyclist)",       "agent.type = vru"),
    ("cyclist",                 "agent.type = cyclist"),
    ("stop sign",               "obj.type = stop_sign"),
    ("yield sign",              "obj.type = yield_sign"),
    ("crosswalk environment",   "env.type = crosswalk"),
    ("intersection environment", "env.type = intersection"),
    ("roundabout environment",  "env.type = roundabout"),
]
headline = pd.DataFrame(
    [(label, ds.count(q), entity_count(q)) for label, q in primitives],
    columns=["category", "clips", "entities"],
).sort_values("entities", ascending=False, ignore_index=True)
headline


## Distribution of agent types

`group_by(query, key)` buckets matches by the value of `key`
and returns **clip counts**. Pair it with
`entities_by_attr(query, attr)` to see entity counts in the
same buckets.


In [ ]:
agent_query = "agent.type = vehicle or agent.type = vru or agent.type = animal"

clips_by_type    = ds.group_by(agent_query, key="agent.type")
entities_by_type = entities_by_attr(agent_query, attr="type")

agent_df = pd.DataFrame({
    "clips":    pd.Series(clips_by_type),
    "entities": pd.Series(entities_by_type),
}).fillna(0).astype(int)
agent_df = agent_df.sort_values("entities", ascending=True)

fig, ax = plt.subplots(figsize=(9, 0.4 * len(agent_df) + 1.5))
y = range(len(agent_df))
ax.barh([i + 0.2 for i in y], agent_df["clips"],    height=0.4,
        color="#3b6ea5", label="clips")
ax.barh([i - 0.2 for i in y], agent_df["entities"], height=0.4,
        color="#8aa6c8", label="entities")
ax.set_yticks(list(y))
ax.set_yticklabels(agent_df.index)
ax.set_xlabel("count")
ax.set_title("Agent types — clip coverage vs entity prevalence")
ax.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.show()

agent_df


## Distribution of ego actions


In [ ]:
ego_dist = ds.group_by(
    "ego.action.type = drive or ego.action.type = stop or ego.action.type = decel "
    "or ego.action.type = yield or ego.action.type = turn_left "
    "or ego.action.type = turn_right or ego.action.type = change_lane",
    key="ego.action.type",
)
s = pd.Series(ego_dist, name="clips").sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 0.45 * max(len(s), 4) + 1.5))
s.plot(kind="barh", ax=ax, color="#4f7d4a")
ax.set_xlabel("clips")
ax.set_ylabel("")
ax.set_title("Clips per ego-action type")
for i, v in enumerate(s.values):
    ax.text(v + max(s.values) * 0.01, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.show()


## Distribution of environment types


In [ ]:
env_dist = ds.group_by(
    "env.type = road or env.type = intersection or env.type = crosswalk "
    "or env.type = sidewalk or env.type = roundabout or env.type = cycle_lane "
    "or env.type = tunnel",
    key="env.type",
)
s = pd.Series(env_dist, name="clips").sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 0.45 * max(len(s), 4) + 1.5))
s.plot(kind="barh", ax=ax, color="#8a5a3b")
ax.set_xlabel("clips")
ax.set_ylabel("")
ax.set_title("Clips per environment type")
for i, v in enumerate(s.values):
    ax.text(v + max(s.values) * 0.01, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.show()


## Composite questions

DSL queries compose, so you can pose questions that combine
primitives — for example, "of the clips that contain a red
light, what fraction also contain an ego stop?"


In [ ]:
def cond_row(label: str, pop_query: str, sub_query: str):
    """One row of the composite-questions table: (label, n_sub, n_pop, n_sub/n_pop)."""
    pop = ds.count(pop_query)
    sub = ds.count(sub_query)
    return (label, sub, pop, sub / max(pop, 1))


composite = pd.DataFrame(
    [
        cond_row(
            "ego stops at red light",
            "light.color = red",
            "light.color = red and ego.action = stop",
        ),
        cond_row(
            "ego stops at stop sign",
            "obj.type = stop_sign",
            "obj.type = stop_sign and ego.action.type = stop",
        ),
        cond_row(
            "ego yields/stops at yield sign",
            "obj.type = yield_sign",
            "obj.type = yield_sign and (ego.action.type = yield or ego.action.type = stop)",
        ),
        cond_row(
            "ego decel/stops at crosswalk with ped",
            "env.type = crosswalk and agent.type = ped",
            "env.type = crosswalk and agent.type = ped "
            "and (ego.action.type = decel or ego.action.type = stop)",
        ),
        cond_row(
            "ped at crosswalk (vs any ped)",
            "agent.type = ped",
            "agent.type = ped and env.type = crosswalk",
        ),
        cond_row(
            "cyclist on cycle/bike lane (vs any cyclist)",
            "agent.type = cyclist",
            "agent.type = cyclist and (env.type = cycle_lane or env.type = bike_lane)",
        ),
        cond_row(
            "ego turn at intersection",
            "env.type = intersection",
            "env.type = intersection and "
            "(ego.action.type = turn_left or ego.action.type = turn_right or ego.action.type = uturn)",
        ),
        cond_row(
            "yellow light — ego could have cleared",
            "light.color = yellow",
            "light(color = yellow, could_have_cleared = true)",
        ),
        cond_row(
            "jaywalking ped (vs any ped)",
            "agent.type = ped",
            "agent(type = ped, action(jaywalk = true))",
        ),
    ],
    columns=["question", "matching clips", "denominator (clips)", "ratio"],
)
composite.style.format({"ratio": "{:.0%}"})


## Action-flag prevalence

Agent action records carry boolean flags that mark notable
patterns: `jaywalk`, `erratic`, `aggressive`, `illegal`,
`aborted`, `unprotected`. The rate below is
`agents-with-flag-true / agents-of-that-type`. Use it to find
rare-event scenarios and which agent type contributes most.

In [ ]:
FLAGS = ["jaywalk", "erratic", "aggressive", "illegal", "aborted", "unprotected"]
AGENT_TYPES_FOR_FLAGS = ["ped", "vehicle", "cyclist"]

flag_rows = []
for atype in AGENT_TYPES_FOR_FLAGS:
    total = entity_count(f"agent.type = {atype}")
    for flag in FLAGS:
        n = entity_count(f"agent(type = {atype}, action({flag} = true))")
        flag_rows.append((atype, flag, n, total, n / max(total, 1)))

flag_df = pd.DataFrame(
    flag_rows, columns=["agent type", "flag", "matching entities", "total entities", "rate"]
)
flag_df.style.format({"rate": "{:.1%}"})

In [ ]:
pivot = flag_df.pivot(index="flag", columns="agent type", values="rate")
fig, ax = plt.subplots(figsize=(9, 4))
pivot.plot(kind="bar", ax=ax, color=["#3b6ea5", "#8a5a3b", "#4f7d4a"])
ax.set_ylabel("fraction of agents (entity-weighted)")
ax.set_xlabel("")
ax.set_title("Action-flag prevalence by agent type")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Agent × environment co-occurrence

A heatmap of clip counts for each `agent.type` × `env.type`
pairing. Use it to spot under-represented combinations
(e.g. cyclist + tunnel, pedestrian + roundabout) and the
densest scenarios.

In [ ]:
AGENTS_FOR_MATRIX = ["ped", "cyclist", "vehicle"]
ENVS_FOR_MATRIX = ["road", "intersection", "crosswalk", "cycle_lane", "roundabout", "sidewalk"]

matrix = np.zeros((len(AGENTS_FOR_MATRIX), len(ENVS_FOR_MATRIX)), dtype=int)
for i, a in enumerate(AGENTS_FOR_MATRIX):
    for j, e in enumerate(ENVS_FOR_MATRIX):
        matrix[i, j] = ds.count(f"agent.type = {a} and env.type = {e}")

fig, ax = plt.subplots(figsize=(8, 3.5))
im = ax.imshow(matrix, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(ENVS_FOR_MATRIX)), labels=ENVS_FOR_MATRIX, rotation=30, ha="right")
ax.set_yticks(range(len(AGENTS_FOR_MATRIX)), labels=AGENTS_FOR_MATRIX)
ax.set_title("Clip count by agent.type × env.type")
threshold = matrix.max() / 2 if matrix.max() else 0
for i in range(len(AGENTS_FOR_MATRIX)):
    for j in range(len(ENVS_FOR_MATRIX)):
        ax.text(
            j, i, matrix[i, j],
            ha="center", va="center",
            color="white" if matrix[i, j] > threshold else "black",
            fontsize=10,
        )
fig.colorbar(im, ax=ax, shrink=0.7, label="clips")
plt.tight_layout()
plt.show()

## Environmental conditions

Per-clip weather, lighting, and road-state via the `cond`
entity. Tells you the mix of operating conditions in the
corpus.

In [ ]:
cond_dist = ds.group_by(
    "cond.type = clear or cond.type = wet or cond.type = snowy "
    "or cond.type = construction or cond.type = temp_marked",
    key="cond.type",
)
s = pd.Series(cond_dist, name="clips").sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 0.45 * max(len(s), 4) + 1.5))
s.plot(kind="barh", ax=ax, color="#6b5a8a")
ax.set_xlabel("clips")
ax.set_ylabel("")
ax.set_title("Clips per environmental condition")
for i, v in enumerate(s.values):
    ax.text(v + max(s.values) * 0.01, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.show()

## Scene density (agents per clip)

How crowded is a typical clip? Histogram of the agent count
in each clip's annotation tree. Pair it with the headline
counts above to filter for sparse vs dense scenarios.

In [ ]:
agent_counts = pd.Series(
    [len(seq.annotation.annotation.agents) for seq in ds],
    name="agents per clip",
)
max_count = int(agent_counts.max())
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(
    agent_counts,
    bins=range(0, max_count + 2),
    color="#3b6ea5",
    edgecolor="white",
)
ax.set_xlabel("agents per clip")
ax.set_ylabel("number of clips")
ax.set_title(
    f"Scene density — {len(ds)} clips, "
    f"median {agent_counts.median():.0f}, "
    f"mean {agent_counts.mean():.1f}, "
    f"max {max_count}"
)
plt.tight_layout()
plt.show()